# Uber Pickup Clustering and Demand Forecasting

**Course task:** cluster Uber pickup locations, examine how demand changes by hour and date, and use the resulting zones for short-term forecasting.

The data covers New York City from September 1–30, 2014. The original CSV is supplied separately and is intentionally not committed to this repository. To rerun the notebook, place `uber-data.csv` in this folder and run the pipeline cell below.

## 1. Reproducible analysis

The complete source code is in `analyze_uber_rides.py`. The script converts coordinates to approximate local kilometres, compares k = 2–10, fits the selected MiniBatch K-means model to all trips, creates figures, and evaluates a chronological demand forecast.


In [1]:
!python analyze_uber_rides.py

Rows: 1,028,136
Period: 2014-09-01 00:00:00 to 2014-09-30 22:59:00
Missing values: 0
Exact repeated rows retained: 24,037
Selected k: 8 (combined rank across three internal indices)
Busiest hour: 18:00
Busiest date: 2014-09-13
Holdout overall WAPE: 14.53%

## 2. Data and model selection

There are 1,028,136 trips, no missing values, and 24,037 exact repeated rows. Since there is no trip ID, repeated rows are retained rather than assumed to be accidental duplicates.

The three internal metrics do not agree on one value of *k*. Silhouette is highest for two very broad clusters, while Calinski–Harabasz and Davies–Bouldin both support *k* = 8. Combining their ranks gives *k* = 8, which preserves more useful geographic detail for the zone-level time analysis. The inertia curve is also provided as an elbow diagnostic.

| k | Silhouette | Calinski–Harabasz | Davies–Bouldin | Rank sum |
|---:|---:|---:|---:|---:|
| 2 | 0.6647 | 10,500.1 | 0.8951 | 19 |
| 3 | 0.4074 | 11,975.0 | 0.8045 | 14 |
| 4 | 0.3831 | 11,446.6 | 0.8131 | 18 |
| 5 | 0.3714 | 11,443.8 | 0.8838 | 22 |
| 6 | 0.4310 | 13,459.5 | 0.7199 | 7 |
| 7 | 0.3903 | 12,769.1 | 0.7642 | 12 |
| **8** | **0.4641** | **13,804.9** | **0.6954** | **4** |
| 9 | 0.3031 | 10,994.3 | 0.8485 | 24 |
| 10 | 0.3589 | 13,086.5 | 0.7865 | 15 |

![Cluster-selection diagnostics](figures/cluster_quality.png)

## 3. Geographic clusters

MiniBatch K-means assigns every trip to one of eight zones. The points below are a random sample for readability; the model itself was fitted to all records. The area names are approximate interpretations of the centers, not official neighbourhood boundaries.

| Cluster | Center latitude | Center longitude | Trips | Share | Approximate area |
|---:|---:|---:|---:|---:|---|
| 0 | 40.7427 | -73.9920 | 353,560 | 34.39% | Midtown / central Manhattan |
| 1 | 40.7078 | -74.0006 | 174,228 | 16.95% | Lower Manhattan |
| 2 | 40.6595 | -73.7611 | 30,718 | 2.99% | JFK Airport area |
| 3 | 40.8126 | -73.9489 | 42,668 | 4.15% | Upper Manhattan / Harlem |
| 4 | 40.7667 | -73.9692 | 264,448 | 25.72% | Upper Midtown / Central Park area |
| 5 | 40.7729 | -73.8552 | 40,275 | 3.92% | LaGuardia Airport area |
| 6 | 40.6941 | -74.2070 | 10,127 | 0.98% | Newark / western New Jersey area |
| 7 | 40.6933 | -73.9512 | 112,112 | 10.90% | Brooklyn |

![Pickup clusters](figures/pickup_clusters.png)

## 4. Demand by time

These plots show the number of pickups in each cluster. Demand peaks at 18:00 across the full dataset, with 75,040 trips in that hour over the month. September 13 is the busiest date, with 43,205 trips.

![Hourly demand by cluster](figures/hourly_demand_by_cluster.png)

![Daily demand by cluster](figures/daily_demand_by_cluster.png)

## 5. Forecasting

Demand is aggregated to hourly counts per cluster. A weekday-and-hour average is estimated from September 1–23 and tested on September 24–30. The same method, refitted to the full month, produces a 168-hour forecast for October 1–7 in `forecast_october_1_to_7.csv`.

The holdout overall WAPE is **14.53%**. Error varies by zone; small zones can have a high percentage error even when their absolute error is low.

| Cluster | MAE | RMSE | WAPE |
|---:|---:|---:|---:|
| 0 | 71.76 | 103.28 | 15.15% |
| 1 | 26.21 | 39.11 | 10.79% |
| 2 | 10.35 | 14.39 | 25.07% |
| 3 | 9.72 | 12.84 | 15.65% |
| 4 | 54.88 | 82.55 | 14.74% |
| 5 | 9.95 | 13.38 | 18.48% |
| 6 | 4.08 | 5.51 | 29.39% |
| 7 | 20.74 | 27.91 | 12.23% |

![Observed and forecast demand](figures/forecast_holdout_week.png)

## 6. Conclusion and limitations

The location data can be partitioned into interpretable geographic zones, with central Manhattan accounting for the largest pickup shares. Demand differs by zone and time of day, so zone-level analysis is more informative than treating New York as one location. A weekday/hour profile provides a useful short-term baseline.

The dataset covers one month only. It cannot support reliable conclusions about annual seasonality, holidays, or long-term demand changes. The cluster metrics measure geometric separation, while the clusters are operational zones rather than official neighbourhoods. Cluster labels are arbitrary identifiers.